# Module 8 – Swap Pricer

## Objective

This notebook demonstrates how discounted cash flows are aggregated into the Net Present Value (NPV) of an Interest Rate Swap.

Pricing Pipeline

Market Quotes
→ Bootstrapper
→ Yield Curve
→ Schedule Generator
→ Cash Flow Generator
→ Discounting
→ Swap Pricer
→ Swap Valuation

For this stage, only the fixed leg is valued.

Swap PV = Σ Discounted Cash Flows

In [1]:
from datetime import date

from src.cashflows.cashflow_generator import CashFlowGenerator
from src.common.enums import (
    DayCountConvention,
    FloatingIndex,
    PayReceive,
    PaymentFrequency,
)
from src.curves.bootstrapper import Bootstrapper
from src.market_data.market_instrument import MarketInstrument
from src.market_data.market_quote_collection import MarketQuoteCollection
from src.pricing.discounting import Discounting
from src.pricing.swap_pricer import SwapPricer
from src.products.fixed_leg import FixedLeg
from src.products.floating_leg import FloatingLeg
from src.products.interest_rate_swap import InterestRateSwap
from src.products.trade import Trade
from src.schedule.schedule_generator import ScheduleGenerator

## Step 1

Build a zero-coupon yield curve from market quotes.

In [4]:
quotes = MarketQuoteCollection()

quotes.add(MarketInstrument("Deposit", "1M", 0.048))
quotes.add(MarketInstrument("Deposit", "3M", 0.050))
quotes.add(MarketInstrument("Deposit", "6M", 0.052))
quotes.add(MarketInstrument("Deposit", "1Y", 0.055))
quotes.add(MarketInstrument("Deposit", "2Y", 0.057))
quotes.add(MarketInstrument("Deposit", "3Y", 0.059))

curve = Bootstrapper(
    valuation_date=date(2026, 1, 1),
    market_quotes=quotes,
).build()

curve.summary()

Yield Curve
Valuation Date : 2026-01-01

Tenor        Years       Zero Rate       Discount Factor
------------------------------------------------------------------------
1M          0.0833        4.8000%              0.996008
3M          0.2500        5.0000%              0.987578
6M          0.5000        5.2000%              0.974335
1Y          1.0000        5.5000%              0.946485
2Y          2.0000        5.7000%              0.892258
3Y          3.0000        5.9000%              0.837780
------------------------------------------------------------------------
Interpolation : Linear
Curve Points  : 6


## Step 2

Create a vanilla fixed-for-floating interest rate swap.

In [2]:
trade = Trade(
    trade_id="IRS001",
    counterparty="ABC Bank",
    currency="USD",
    effective_date=date(2026, 1, 1),
    maturity_date=date(2029, 1, 1),
)

fixed_leg = FixedLeg(
    notional=1_000_000,
    fixed_rate=0.05,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.RECEIVE,
)

floating_leg = FloatingLeg(
    notional=1_000_000,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.PAY,
    floating_index=FloatingIndex.SOFR,
)

swap = InterestRateSwap(
    trade=trade,
    fixed_leg=fixed_leg,
    floating_leg=floating_leg,
)

## Step 3

Generate cash flows and discount them using the yield curve.

In [5]:
schedule = ScheduleGenerator().generate(swap)

cashflows = CashFlowGenerator().generate(
    swap,
    schedule,
)

discounted_cashflows = Discounting().discount(
    cashflows,
    curve,
)

for cashflow in discounted_cashflows:
    print(cashflow)

2027-01-01 | DF=0.946485 | CF=50,000.00 | PV=47,324.26
2028-01-01 | DF=0.892258 | CF=50,000.00 | PV=44,612.90
2029-01-01 | DF=0.837780 | CF=50,136.99 | PV=42,003.75


## Step 4

Aggregate the discounted cash flows into the swap valuation.

In [6]:
valuation = SwapPricer().value(
    swap,
    discounted_cashflows,
)

print(valuation)
print()
print(f"Present Value : {valuation.present_value:,.2f} {valuation.currency}")

SwapValuation(trade_id='IRS001', currency='USD', present_value=133,940.91)

Present Value : 133,940.91 USD


## Summary

In this module, the pricing engine aggregates discounted cash flows into a single present value (PV).

Completed Pricing Pipeline

Market Quotes
→ Yield Curve
→ Schedule Generation
→ Cash Flow Generation
→ Discounting
→ Swap Pricer
→ Swap Valuation

The next module will introduce **DV01/PV01 Risk Sensitivity**, where we measure how the swap value changes in response to movements in interest rates.